In [46]:
# V6-01 — setup, paths, load linking_v5 structural CSV

from pathlib import Path
import pandas as pd
import numpy as np
import re

pd.set_option("display.max_columns", 180)
pd.set_option("display.width", 240)

ROOT = Path("/Users/davekokel/Projects/carp_v2")
BASE = ROOT / "seed_kits" / "legacy_wrangling_v2"
RAW = BASE / "raw"
WORKING = BASE / "working"

STRUCT_PATH = WORKING / "output_from_linking_v5.csv"

print("ROOT:", ROOT)
print("STRUCT_PATH:", STRUCT_PATH)

if not STRUCT_PATH.exists():
    raise FileNotFoundError(f"Missing structural CSV: {STRUCT_PATH}")

df_struct = pd.read_csv(STRUCT_PATH)

print("\nV6-01 — df_struct loaded")
print("shape:", df_struct.shape)
print("columns:", list(df_struct.columns))
print("unique roi_dir:", df_struct["roi_dir"].nunique())

ROOT: /Users/davekokel/Projects/carp_v2
STRUCT_PATH: /Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling_v2/working/output_from_linking_v5.csv

V6-01 — df_struct loaded
shape: (1082, 46)
columns: ['date_experiment', 'fish', 'roi_rel', 'roi_name', 'roi_tiffs', 'roi_dir', 'dataset', 'experiment_folder', 'roi_path_date_yyyymmdd', 'fish_folder', 'roi_folder', 'fish_id', 'fish_number', 'fish_age_hpf', 'fish_nickname', 'fish_raw_norm', 'roi_anatomy_tokens', 'roi_anatomy', 'fish_folder_patched', 'dataset_slug', 'dataset_slug_norm', 'sheet_slug_norm', 'date_mount_yyyymmdd', 'date_mount', 'Date imaged', 'mount_id', 'ZF female genotype', 'ZF male genotype', 'additional plasmids injected', 'additional mRNAs injected', 'additonal proteins injected', 'additonal dye and chemicals', 'Date born', 'Imaged Locations', 'Unique Targets with blanks', 'Unique Targets', 'Data location', 'link_source', 'plate_date', 'mount_id_inferred', 'mount_id_source', 'plate_key', 'plate_id_filled', 'slot_id_fill

In [47]:
# V6-02 (fixed) — collapse roi_dir duplicates using only BIOLOGICAL columns

import numpy as np

bio_cols = [
    "roi_dir",
    "experiment_folder",
    "dataset_slug",
    "fish_id",
    "fish_number",
    "fish_age_hpf",
    "roi_anatomy",
    "roi_tiffs",
    "date_experiment",
]

# imaging-sheet noise columns to drop BEFORE dedupe
noise_cols = [
    "mount_id",
    "additional plasmids injected",
    "additional mRNAs injected",
    "additonal proteins injected",
    "additonal dye and chemicals",
    "Date born",
    "Time mounted",
    "Mounting Orientation",
    "Date screened/Initial feedback",
    "Date imaged",
    "Time placed in scope",
    "Start of imaging time",
    "End of imaging time",
    "Imaged Locations",
    "Unique Targets with blanks",
    "Unique Targets",
    "Data location",
    "Dataset size (GB) - raw data only",
    "Camera Filters",
    "JSON excite map for ZF male",
    "JSON excite map for ZF female",
    "JSON excite map for plasmid",
    "JSON excite map for mRNA",
    "comments",
    "Data evaluation comments",
]

# drop the noise columns *if present*
drop_now = [c for c in noise_cols if c in df_struct.columns]
df_struct_clean = df_struct.drop(columns=drop_now, errors="ignore").copy()

# now detect duplicates
dups = df_struct_clean[df_struct_clean.duplicated("roi_dir", keep=False)]
print("V6-02 — duplicate roi_dir rows (after noise drop):", len(dups))

if len(dups) > 0:
    # check differences only in biological columns
    nunq = (
        dups.groupby("roi_dir")[bio_cols]
        .nunique(dropna=False)
    )

    varying_bio = [
        c for c in bio_cols
        if c != "roi_dir" and (nunq[c] > 1).any()
    ]

    if varying_bio:
        print("\nV6-02 — ERROR: duplicates differ in BIOLOGICAL columns:", varying_bio)
        raise ValueError("Structural duplicates conflict in biological fields — cannot resolve automatically.")

    print("\nV6-02 — duplicates differ only in imaging-sheet noise → collapsing by first occurrence.")

    df_struct_clean = (
        df_struct_clean
        .sort_values(["roi_dir"])
        .drop_duplicates("roi_dir", keep="first")
        .reset_index(drop=True)
    )

df_struct = df_struct_clean

print("\nV6-02 — AFTER CLEAN COLLAPSE:")
print("  rows:", len(df_struct))
print("  unique roi_dir:", df_struct["roi_dir"].nunique())

V6-02 — duplicate roi_dir rows (after noise drop): 208

V6-02 — duplicates differ only in imaging-sheet noise → collapsing by first occurrence.

V6-02 — AFTER CLEAN COLLAPSE:
  rows: 976
  unique roi_dir: 976


In [48]:
# V6-03 — load parent map v5, injected maps, constructs, fluors, tags, alias

PARENT_MAP_XLSX = RAW / "Unique_parent_names__mom_dad_combined__preview_dqm_v5.xlsx"
PARENT_MAP_CSV  = RAW / "Unique_parent_names__mom_dad_combined__preview_dqm_v5.csv"

if PARENT_MAP_XLSX.exists():
    parent_map = pd.read_excel(PARENT_MAP_XLSX)
    parent_src = "xlsx"
elif PARENT_MAP_CSV.exists():
    parent_map = pd.read_csv(PARENT_MAP_CSV)
    parent_src = "csv"
else:
    raise FileNotFoundError("Missing parent_map v5: expected either XLSX or CSV.")

INJECTED_PLASMID_PATH = RAW / "Unique_injected_plasmid__preview_dqm.xlsx"
INJECTED_RNA_PATH     = RAW / "Unique_injected_rna__preview_dqm.xlsx"
PLASMIDS_JANELIA_PATH = RAW / "plasmids_janelia_googlesheet.xlsx"

AUTO = ROOT / "seed_kits" / "2025-11-15-121231-autoload"
CONSTRUCTS_PATH = AUTO / "constructs_plasmid.csv"
FLUORS_PATH     = AUTO / "fluors.csv"
TAGS_PATH       = AUTO / "tags.xlsx"
ALIAS_PATH      = AUTO / "alias.csv"

# load everything
injected_plasmid = pd.read_excel(INJECTED_PLASMID_PATH)
injected_rna     = pd.read_excel(INJECTED_RNA_PATH)
plasmids_sheet   = pd.read_excel(PLASMIDS_JANELIA_PATH)

constructs       = pd.read_csv(CONSTRUCTS_PATH)
fluors           = pd.read_csv(FLUORS_PATH)
tags_cat         = pd.read_excel(TAGS_PATH)
alias            = pd.read_csv(ALIAS_PATH)

print("\nV6-03 — Loaded reference catalogs:")
print("  parent_map:", parent_map.shape, f"({parent_src})")
print("  injected_plasmid:", injected_plasmid.shape)
print("  injected_rna:", injected_rna.shape)
print("  plasmids_sheet:", plasmids_sheet.shape)
print("  constructs:", constructs.shape)
print("  fluors:", fluors.shape)
print("  tags:", tags_cat.shape)
print("  alias:", alias.shape)


V6-03 — Loaded reference catalogs:
  parent_map: (52, 5) (csv)
  injected_plasmid: (25, 2)
  injected_rna: (44, 2)
  plasmids_sheet: (65, 6)
  constructs: (286, 12)
  fluors: (28, 5)
  tags: (10, 5)
  alias: (6, 3)


In [49]:
# V6-03b — apply parent_hole_patch_v6.csv overrides

from pathlib import Path

if "ROOT" not in globals():
    ROOT = Path("/Users/davekokel/Projects/carp_v2")
RAW = ROOT / "seed_kits" / "legacy_wrangling_v2" / "raw"

PATCH_PATH = RAW / "parent_hole_patch_v6.csv"
print("V6-03b — PATCH_PATH:", PATCH_PATH)

if "parent_map" not in globals():
    raise NameError("V6-03b: parent_map not found — run V6-03 first.")

if not PATCH_PATH.exists():
    raise FileNotFoundError(f"V6-03b: patch file not found at {PATCH_PATH}")

patch = pd.read_csv(PATCH_PATH, dtype=str)

expected_cols = ["parent_fish_name", "plasmid_base_code", "allele", "injected_rna", "injected_plasmid"]
missing_cols = [c for c in expected_cols if c not in patch.columns]
if missing_cols:
    raise KeyError(f"V6-03b: patch CSV missing columns: {missing_cols}")

# Coerce parent_map columns to string so concat is clean
for c in expected_cols:
    if c in parent_map.columns:
        parent_map[c] = parent_map[c].astype("string")

patch = patch[expected_cols].copy()
patch = patch.astype("string")

before_rows = len(parent_map)
before_unique = parent_map["parent_fish_name"].nunique()

parent_map = pd.concat([parent_map, patch], ignore_index=True)

# For any duplicate parent_fish_name, keep the last (patch wins)
parent_map = (
    parent_map
    .drop_duplicates(subset=["parent_fish_name"], keep="last")
    .reset_index(drop=True)
)

after_rows = len(parent_map)
after_unique = parent_map["parent_fish_name"].nunique()

print("V6-03b — parent_map patch applied")
print("  rows before:", before_rows, " unique parents:", before_unique)
print("  rows after: ", after_rows,  " unique parents:", after_unique)

print("\nV6-03b — patched parents sample (skittle / mem-mito / mem-histone etc.):")
print(
    parent_map[
        parent_map["parent_fish_name"].str.contains(
            "skittle|mem-mito|mem-histone|mitomSG|er-mSG|peroxi|mem-kinetocore",
            case=False,
            na=False,
        )
    ].head(20)
)

V6-03b — PATCH_PATH: /Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling_v2/raw/parent_hole_patch_v6.csv
V6-03b — parent_map patch applied
  rows before: 52  unique parents: 52
  rows after:  52  unique parents: 52

V6-03b — patched parents sample (skittle / mem-mito / mem-histone etc.):
       parent_fish_name plasmid_base_code   allele     injected_rna injected_plasmid
41             mem-mito           pDQM082      315          MGCO-01             <NA>
42          mem-histone   pDQM005,pDQM133  302,324             <NA>             <NA>
43             skittlez           pDQM034      309             <NA>             <NA>
44             skittles           pDQM034      309             <NA>             <NA>
45              mitomSG           pDQM005      301          MGCO-01             <NA>
46  er-mSG_mem-mChilada           pDQM082      315  MGCO-04,MGCO-01             <NA>
47  mem-mchilada_er-mSG           pDQM082      315  MGCO-01,MGCO-04             <NA>
50               perox

In [50]:
# V6-04 — init df_enrich + normalize imaging treatment columns

import numpy as np

df_enrich = df_struct.copy()

print("V6-04 — starting df_enrich shape:", df_enrich.shape)

# normalize imaging-sheet treatment columns → treatment_*_names_sheet
rename_map = {
    "additional plasmids injected": "treatment_plasmid_names_sheet",
    "additional mRNAs injected": "treatment_rna_names_sheet",
    "additonal proteins injected": "treatment_protein_names_sheet",
    "additonal dye and chemicals": "treatment_dye_names_sheet",
}
cols_present = {c for c in df_enrich.columns}
for old, new in rename_map.items():
    if old in cols_present and new not in cols_present:
        df_enrich = df_enrich.rename(columns={old: new})

print("V6-04 — df_enrich columns (subset):")
print([c for c in df_enrich.columns if "treatment_" in c or "ZF " in c or "genotype_" in c])

V6-04 — starting df_enrich shape: (976, 35)
V6-04 — df_enrich columns (subset):
['ZF female genotype', 'ZF male genotype']


In [51]:
# V6-05 — genotype + slug-level parent map (skittles, mem-mito, mem-histone, etc.)

import re

if "parent_map" not in globals():
    raise NameError("V6-05: parent_map not loaded; run V6-03 first.")

df_enrich = df_enrich.copy()
pm = parent_map.copy()

# 1) normalize parent_fish_name → parent_slug_norm
def _norm_label(s):
    if pd.isna(s):
        return None
    s = str(s).strip().lower()
    s = re.sub(r"^\d{8}[_-]", "", s)          # strip leading 8-digit date
    s = re.sub(r"\(.*?\)", "", s)             # remove parentheses
    s = re.sub(r"[^a-z0-9]+", "", s)          # keep alnum
    s = s.strip()
    return s or None

pm["parent_slug_norm"] = pm["parent_fish_name"].apply(_norm_label)

# 2) experiment_folder → exp_slug_norm
if "experiment_folder" not in df_enrich.columns:
    raise KeyError("V6-05: df_enrich missing 'experiment_folder' column.")
df_enrich["exp_slug_norm"] = df_enrich["experiment_folder"].apply(_norm_label)

print("V6-05 — slug coverage:")
print("  exp_slug_norm unique:", df_enrich["exp_slug_norm"].nunique())
print("  parent_slug_norm unique:", pm["parent_slug_norm"].nunique())

# 3) aggregate parent map by slug → geno + injected names
def _agg_nonempty(series: pd.Series):
    vals = [
        str(x).strip()
        for x in series
        if pd.notna(x) and str(x).strip() not in ("", "nan", "n/a", "na")
    ]
    if not vals:
        return None
    seen = set()
    out = []
    for v in vals:
        if v not in seen:
            seen.add(v)
            out.append(v)
    return "|".join(out)

parent_slug_agg = (
    pm.groupby("parent_slug_norm", dropna=True)
      .agg({
          "plasmid_base_code": _agg_nonempty,
          "allele": _agg_nonempty,
          "injected_rna": _agg_nonempty,
          "injected_plasmid": _agg_nonempty,
      })
      .reset_index()
      .rename(columns={
          "plasmid_base_code": "geno_base_codes_v6",
          "allele": "geno_alleles_v6",
          "injected_rna": "inj_rna_names_v6",
          "injected_plasmid": "inj_plasmid_names_v6",
      })
)

print("\nV6-05 — parent_slug_agg sample:")
print(parent_slug_agg.head(15))

# 4) join slug-agg onto df_enrich
df_enrich = df_enrich.merge(
    parent_slug_agg,
    how="left",
    left_on="exp_slug_norm",
    right_on="parent_slug_norm",
)

print("\nV6-05 — df_enrich after slug join:")
print("  shape:", df_enrich.shape)
print("  added columns:",
      [c for c in ["parent_slug_norm", "geno_base_codes_v6", "geno_alleles_v6",
                   "inj_rna_names_v6", "inj_plasmid_names_v6"]
       if c in df_enrich.columns])

# 5) prepare target cols as object dtypes before patching
for col in ["genotype_base_codes", "genotype_allele_codes", "treatment_rna_names_sheet"]:
    if col not in df_enrich.columns:
        df_enrich[col] = np.nan
    df_enrich[col] = df_enrich[col].astype("object")

def _is_empty(s: pd.Series) -> pd.Series:
    return s.isna() | (s.astype(str).str.strip().isin(["", "None", "nan"]))

mask_geno_missing   = _is_empty(df_enrich["genotype_base_codes"])
mask_allele_missing = _is_empty(df_enrich["genotype_allele_codes"])

mask_geno_has_v6   = df_enrich["geno_base_codes_v6"].notna()
mask_allele_has_v6 = df_enrich["geno_alleles_v6"].notna()

to_patch_geno   = mask_geno_missing & mask_geno_has_v6
to_patch_allele = mask_allele_missing & mask_allele_has_v6

df_enrich.loc[to_patch_geno,   "genotype_base_codes"]   = df_enrich.loc[to_patch_geno,   "geno_base_codes_v6"].astype("object")
df_enrich.loc[to_patch_allele, "genotype_allele_codes"] = df_enrich.loc[to_patch_allele, "geno_alleles_v6"].astype("object")

print("\nV6-05 — genotype patch counts:")
print("  rows with empty genotype_base_codes:", int(mask_geno_missing.sum()))
print("  rows with v6 geno_base_codes_v6:", int(mask_geno_has_v6.sum()))
print("  rows patched (base codes):", int(to_patch_geno.sum()))
print("  rows patched (alleles):   ", int(to_patch_allele.sum()))

# 6) patch treatment_rna_names_sheet from inj_rna_names_v6 where empty
mask_rna_empty = _is_empty(df_enrich["treatment_rna_names_sheet"])
mask_rna_has_v6 = df_enrich["inj_rna_names_v6"].notna()
to_patch_rna = mask_rna_empty & mask_rna_has_v6

df_enrich.loc[to_patch_rna, "treatment_rna_names_sheet"] = df_enrich.loc[to_patch_rna, "inj_rna_names_v6"].astype("object")

print("\nV6-05 — treatment_rna_names_sheet patch:")
print("  empty before:", int(mask_rna_empty.sum()))
print("  with inj_rna_names_v6:", int(mask_rna_has_v6.sum()))
print("  patched rows:", int(to_patch_rna.sum()))

V6-05 — slug coverage:
  exp_slug_norm unique: 32
  parent_slug_norm unique: 41

V6-05 — parent_slug_agg sample:
                               parent_slug_norm geno_base_codes_v6      geno_alleles_v6 inj_rna_names_v6 inj_plasmid_names_v6
0                                           abe            pDQM034                  309             <NA>                 <NA>
1                                           ben            pDQM034                  310             <NA>                 <NA>
2                                     casperrnf               <NA>                 <NA>             <NA>                 <NA>
3                                         chris            pDQM034                  317             <NA>                 <NA>
4                                  csppiglet14a               <NA>                 <NA>             <NA>                 <NA>
5                                        dennis            pDQM036                  318             <NA>                 <NA>
6    

In [53]:
# V6-07c — treatment RNA → marker rollup → organelles union (from parent_hole_patch_v6)

import pandas as pd

# sanity: required inputs
for name in ["df_enrich", "constructs", "tags_cat", "fluors", "injected_rna"]:
    if name not in globals():
        raise NameError(f"V6-07c: {name} not found; run earlier V6 cells first.")

# ─────────────────────────────────────────────
# 0) Rebuild constructs_ft (plasmid_code → fluor/tag/localization)
# ─────────────────────────────────────────────
print("V6-07c — rebuilding constructs_ft")

required_construct_cols = {"plasmid_code", "fluor_code", "tag_code"}
missing_c = required_construct_cols - set(constructs.columns)
if missing_c:
    raise KeyError(f"V6-07c: constructs is missing columns: {sorted(missing_c)}")

c_small = (
    constructs[["plasmid_code", "fluor_code", "tag_code"]]
    .drop_duplicates()
)

tags_small = (
    tags_cat
    .rename(columns={"nickname": "tag_code", "localization": "tag_localization"})
    [["tag_code", "tag_localization"]]
    .drop_duplicates()
)

fluors_small = (
    fluors
    .rename(columns={"nickname": "fluor_code"})
    [["fluor_code"]]
    .drop_duplicates()
)

constructs_ft = (
    c_small
    .merge(tags_small, on="tag_code", how="left")
    .merge(fluors_small, on="fluor_code", how="left")
)

print("V6-07c — constructs_ft shape:", constructs_ft.shape)
print("V6-07c — constructs_ft columns:", list(constructs_ft.columns))

# ─────────────────────────────────────────────
# 1) Build RNA name → plasmid_base_code map
#     (from Unique_injected_rna__preview_dqm.xlsx)
# ─────────────────────────────────────────────

if "injected_rna" not in globals():
    raise NameError("V6-07c: injected_rna not loaded; run V6-03 first.")

rna_cols_needed = {"injected_rna", "plasmid_base_code"}
missing_rna = rna_cols_needed - set(injected_rna.columns)
if missing_rna:
    raise KeyError(f"V6-07c: injected_rna missing columns: {sorted(missing_rna)}")

rna_map = injected_rna[["injected_rna", "plasmid_base_code"]].copy()
rna_map["injected_rna_norm"] = rna_map["injected_rna"].astype(str).str.strip()
rna_map["plasmid_base_code"] = rna_map["plasmid_base_code"].astype(str).str.strip()
rna_map = rna_map.drop_duplicates(subset=["injected_rna_norm"])

print("V6-07c — RNA map rows:", len(rna_map))

# ─────────────────────────────────────────────
# 2) From df_enrich: treatment_rna_names_sheet → RNA basecode → constructs_ft
# ─────────────────────────────────────────────

if "treatment_rna_names_sheet" not in df_enrich.columns:
    print("V6-07c — no treatment_rna_names_sheet column on df_enrich; nothing to do.")
else:
    df_enrich = df_enrich.copy()

    tx = df_enrich[["roi_dir", "treatment_rna_names_sheet"]].copy()
    tx["treatment_rna_names_norm"] = (
        tx["treatment_rna_names_sheet"]
        .astype("string")
        .str.strip()
    )

    # join to RNA map (MGCO-01, MGCO-04, etc → plasmid_base_code)
    tx = tx.merge(
        rna_map[["injected_rna_norm", "plasmid_base_code"]],
        left_on="treatment_rna_names_norm",
        right_on="injected_rna_norm",
        how="left",
    )

    # now plasmid_base_code should match constructs_ft.plasmid_code
    tx = tx.merge(
        constructs_ft[["plasmid_code", "fluor_code", "tag_code", "tag_localization"]],
        left_on="plasmid_base_code",
        right_on="plasmid_code",
        how="left",
    )

    # keep only rows where we actually have a marker
    tx_markers = tx[tx["fluor_code"].notna() | tx["tag_localization"].notna()].copy()

    print("V6-07c — treatment-RNA marker rows:", len(tx_markers))

    # ─────────────────────────────────────────
    # 3) Aggregate per ROI: treatment markers
    # ─────────────────────────────────────────

    def _agg_tx(group: pd.DataFrame) -> pd.Series:
        def _uniq_pipe(vals):
            vals = [
                str(v).strip()
                for v in vals
                if pd.notna(v) and str(v).strip() not in ("", "nan", "<NA>")
            ]
            seen = set()
            out = []
            for v in vals:
                if v not in seen:
                    seen.add(v)
                    out.append(v)
            return "|".join(out) if out else pd.NA

        fluors = _uniq_pipe(group["fluor_code"])
        tags   = _uniq_pipe(group["tag_code"])
        locs   = _uniq_pipe(group["tag_localization"])

        # build fluor(loc) labels
        labels = []
        for f, tloc in zip(
            str(fluors).split("|") if pd.notna(fluors) else [],
            str(locs).split("|") if pd.notna(locs)   else [],
        ):
            f = f.strip()
            tloc = tloc.strip()
            if not f:
                continue
            if tloc:
                labels.append(f"{f}({tloc})")
            else:
                labels.append(f)

        labels_uniq = "|".join(dict.fromkeys(labels)) if labels else pd.NA

        return pd.Series(
            {
                "treatment_marker_fluor_codes": fluors,
                "treatment_marker_tag_codes":   tags,
                "treatment_marker_localizations": locs,
                "treatment_marker_fluor_loc_labels": labels_uniq,
            }
        )

    if tx_markers.empty:
        print("V6-07c — no treatment RNA markers found to aggregate; skipping organelle recompute.")
    else:
        tx_agg = tx_markers.groupby("roi_dir", as_index=False).apply(_agg_tx)

        print("V6-07c — tx_agg sample:")
        print(tx_agg.head(10))

        # join aggregated treatment markers back onto df_enrich
        df_enrich = df_enrich.merge(tx_agg, on="roi_dir", how="left")

        # ─────────────────────────────────────
        # 4) Recompute all_unique_organelles / all_fluor_organelles
        #    as union(genotype + treatment)
        # ─────────────────────────────────────

        def _split_pipe(val):
            if pd.isna(val):
                return []
            s = str(val).strip()
            if s in ("", "nan", "<NA>"):
                return []
            return [p.strip() for p in s.split("|") if p.strip()]

        def _join_unique(items):
            seen = set()
            out = []
            for x in items:
                if x not in seen:
                    seen.add(x)
                    out.append(x)
            return "|".join(out) if out else pd.NA

        new_all_unique = []
        new_all_florg  = []

        for _, row in df_enrich.iterrows():
            # locations
            geno_locs = _split_pipe(row.get("genotype_marker_localizations", pd.NA))
            tx_locs   = _split_pipe(row.get("treatment_marker_localizations", pd.NA))
            all_locs  = _join_unique(geno_locs + tx_locs)
            new_all_unique.append(all_locs)

            # fluor-loc labels
            geno_florg = _split_pipe(row.get("genotype_marker_fusion_labels", pd.NA))
            tx_florg   = _split_pipe(row.get("treatment_marker_fluor_loc_labels", pd.NA))
            all_florg  = _join_unique(geno_florg + tx_florg)
            new_all_florg.append(all_florg)

        df_enrich["all_unique_organelles"] = pd.Series(new_all_unique, index=df_enrich.index, dtype="string")
        df_enrich["all_fluor_organelles"]  = pd.Series(new_all_florg,  index=df_enrich.index, dtype="string")

        print("\nV6-07c — recomputed organelles from genotype + treatment RNA markers.")
        print("  sample rows:")
        print(
            df_enrich[
                [
                    "roi_dir",
                    "dataset_slug",
                    "genotype_marker_localizations",
                    "treatment_marker_localizations",
                    "all_unique_organelles",
                    "all_fluor_organelles",
                ]
            ].head(20)
        )

# quick sanity peek for mem-mito + skittles dataset slugs
for slug in ["20250819_mem_mito", "20250917_mem-mito", "20250513_skittles"]:
    sub = df_enrich[df_enrich["dataset_slug"] == slug]
    if sub.empty:
        continue
    print(f"\nV6-07c — sanity sample for dataset_slug='{slug}':")
    print(
        sub[
            [
                "roi_dir",
                "genotype_base_codes",
                "genotype_allele_codes",
                "treatment_rna_names_sheet",
                "treatment_marker_fluor_loc_labels",
                "all_unique_organelles",
                "all_fluor_organelles",
            ]
        ].head(10)
    )

print("\nV6-07c — done. Next: V6-11 (manual overrides) then V6-10 (QC + write v6 CSVs).")

V6-07c — rebuilding constructs_ft
V6-07c — constructs_ft shape: (286, 4)
V6-07c — constructs_ft columns: ['plasmid_code', 'fluor_code', 'tag_code', 'tag_localization']
V6-07c — RNA map rows: 44
V6-07c — treatment-RNA marker rows: 0
V6-07c — no treatment RNA markers found to aggregate; skipping organelle recompute.

V6-07c — sanity sample for dataset_slug='20250819_mem_mito':


KeyError: "['treatment_marker_fluor_loc_labels', 'all_unique_organelles', 'all_fluor_organelles'] not in index"

In [54]:
# V6-06 — build constructs_ft marker catalog (plasmid → fluor/tag/localization)

# normalize constructs
constructs_small = constructs.rename(columns={"plasmid_code": "plasmid_base_code"})

constructs_small = constructs_small[
    ["plasmid_base_code", "fluor_code", "tag_code", "tag_pos"]
].drop_duplicates()

# normalize tags → tag_localization
tags_small = tags_cat.rename(columns={"nickname": "tag_code", "localization": "tag_localization"})[
    ["tag_code", "tag_localization"]
].drop_duplicates()

# join constructs to tags
constructs_ft = (
    constructs_small
    .merge(tags_small, how="left", on="tag_code")
)

print("V6-06 — constructs_ft sample:")
print(constructs_ft.head(15))

V6-06 — constructs_ft sample:
   plasmid_base_code fluor_code tag_code tag_pos tag_localization
0            pDQM001        mSG      NaN     NaN              NaN
1            pDQM002        mSG      NaN     NaN              NaN
2            pDQM005      tdmSG   2xLynk       N         membrane
3            pDQM006      tdmSG      NaN     NaN              NaN
4            pDQM007      tdmSG      NaN     NaN              NaN
5            pDQM008      tdmSG      NaN     NaN              NaN
6            pDQM009   mScarlet      NaN     NaN              NaN
7            pDQM009     mKate2      NaN     NaN              NaN
8            pDQM009   Electra2      NaN     NaN              NaN
9            pDQM009       mKOK      NaN     NaN              NaN
10           pDQM009      mTFP1      NaN     NaN              NaN
11           pDQM010   mScarlet      NaN     NaN              NaN
12           pDQM010     mKate2      NaN     NaN              NaN
13           pDQM010   Electra2      NaN     N

In [55]:
# V6-07 — genotype marker rollup from genotype_base_codes using constructs_ft

df_enrich = df_enrich.copy()

def _split_codes(val):
    if pd.isna(val):
        return []
    s = str(val)
    if not s.strip():
        return []
    # split on '|' or ',' interchangeably
    parts = re.split(r"[|,]", s)
    return [p.strip() for p in parts if p.strip()]

# explode genotype base codes
geno_rows = []
for idx, row in df_enrich.iterrows():
    codes = _split_codes(row.get("genotype_base_codes", np.nan))
    if not codes:
        continue
    for code in codes:
        geno_rows.append(
            {"roi_dir": row["roi_dir"], "plasmid_base_code": code}
        )

geno = pd.DataFrame(geno_rows)
if geno.empty:
    print("V6-07 — WARNING: no genotype base codes exploded; skipping marker rollup.")
    for col in [
        "genotype_marker_fluor_codes",
        "genotype_marker_tag_codes",
        "genotype_marker_localizations",
        "genotype_marker_fusion_labels",
    ]:
        if col not in df_enrich.columns:
            df_enrich[col] = np.nan
else:
    geno_markers = (
        geno.merge(constructs_ft, how="left", on="plasmid_base_code")
            .dropna(subset=["fluor_code"], how="all")
    )

    def _agg_marker(group):
        fs = _split_codes("|".join(group["fluor_code"].dropna().astype(str)))
        ts = _split_codes("|".join(group["tag_code"].dropna().astype(str)))
        locs = _split_codes("|".join(group["tag_localization"].dropna().astype(str)))

        def _dedup(xs):
            seen = set()
            out = []
            for x in xs:
                if x not in seen:
                    seen.add(x)
                    out.append(x)
            return out

        fs = _dedup(fs)
        ts = _dedup(ts)
        locs = _dedup(locs)

        fusion_labels = []
        for f in fs:
            for loc in locs or ["unknown"]:
                fusion_labels.append(f"{f}({loc})")

        return pd.Series({
            "genotype_marker_fluor_codes": "|".join(fs) if fs else np.nan,
            "genotype_marker_tag_codes": "|".join(ts) if ts else np.nan,
            "genotype_marker_localizations": "|".join(locs) if locs else np.nan,
            "genotype_marker_fusion_labels": "|".join(fusion_labels) if fusion_labels else np.nan,
        })

    geno_agg = geno_markers.groupby("roi_dir", as_index=False).apply(_agg_marker)

    df_enrich = df_enrich.merge(geno_agg, how="left", on="roi_dir")

print("V6-07 — genotype marker rollup sample:")
print(
    df_enrich[
        [
            "roi_dir",
            "genotype_base_codes",
            "genotype_marker_fluor_codes",
            "genotype_marker_localizations",
            "genotype_marker_fusion_labels",
        ]
    ].head(20)
)

V6-07 — genotype marker rollup sample:
                                              roi_dir genotype_base_codes genotype_marker_fluor_codes genotype_marker_localizations genotype_marker_fusion_labels
0   /clusterfs/vast/abcabc/Aang_Foundation/2025072...                 NaN                         NaN                           NaN                           NaN
1   /clusterfs/vast/abcabc/Aang_Foundation/2025072...                 NaN                         NaN                           NaN                           NaN
2   /clusterfs/vast/abcabc/Aang_Foundation/2025072...                 NaN                         NaN                           NaN                           NaN
3   /clusterfs/vast/abcabc/Aang_Foundation/2025072...                 NaN                         NaN                           NaN                           NaN
4   /clusterfs/vast/abcabc/Aang_Foundation/2025072...                 NaN                         NaN                           NaN                    

/var/folders/29/cdrb2nrn01s4d1_n6gvxy5j80000gn/T/ipykernel_98455/3863803533.py:73: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  geno_agg = geno_markers.groupby("roi_dir", as_index=False).apply(_agg_marker)


In [56]:
# V6-08 — treatment RNA → base codes → markers → recompute organelles

df_enrich = df_enrich.copy()

# 1) map treatment_rna_names_sheet → treatment_rna_rna_base_code using injected_rna
inj_rna = injected_rna.copy()

# normalize columns
inj_rna = inj_rna.rename(columns={
    "injected_rna": "treatment_rna_name",
    "plasmid_base_code": "rna_base_code",
})

def _norm_treat_name(s):
    if pd.isna(s):
        return None
    return str(s).strip().lower()

inj_rna["treatment_rna_name_norm"] = inj_rna["treatment_rna_name"].apply(_norm_treat_name)

if "treatment_rna_names_sheet" not in df_enrich.columns:
    df_enrich["treatment_rna_names_sheet"] = np.nan

df_enrich["treatment_rna_name_norm"] = df_enrich["treatment_rna_names_sheet"].apply(_norm_treat_name)

tx_rna_map = inj_rna[["treatment_rna_name_norm", "rna_base_code"]].dropna().drop_duplicates()

df_enrich = df_enrich.merge(
    tx_rna_map,
    how="left",
    on="treatment_rna_name_norm",
)

df_enrich = df_enrich.rename(columns={"rna_base_code": "treatment_rna_rna_base_code"})

print("V6-08 — RNA basecode mapping sample:")
print(
    df_enrich[
        [
            "treatment_rna_names_sheet",
            "treatment_rna_rna_base_code",
        ]
    ].head(20)
)

# 2) treatment markers via constructs_ft

tx_rows = []
for idx, row in df_enrich.iterrows():
    codes = _split_codes(row.get("treatment_rna_rna_base_code", np.nan))
    if not codes:
        continue
    for code in codes:
        tx_rows.append({"roi_dir": row["roi_dir"], "plasmid_base_code": code})

tx_df = pd.DataFrame(tx_rows)

if tx_df.empty:
    print("V6-08 — no treatment RNA base codes found; skipping treatment marker rollup.")
    for col in [
        "treatment_marker_fluor_codes",
        "treatment_marker_tag_codes",
        "treatment_marker_localizations",
        "treatment_marker_fluor_loc_labels",
    ]:
        if col not in df_enrich.columns:
            df_enrich[col] = np.nan
else:
    tx_markers = (
        tx_df.merge(constructs_ft, how="left", on="plasmid_base_code")
             .dropna(subset=["fluor_code"], how="all")
    )

    def _agg_tx(group):
        fs = _split_codes("|".join(group["fluor_code"].dropna().astype(str)))
        ts = _split_codes("|".join(group["tag_code"].dropna().astype(str)))
        locs = _split_codes("|".join(group["tag_localization"].dropna().astype(str)))

        def _dedup(xs):
            seen = set()
            out = []
            for x in xs:
                if x not in seen:
                    seen.add(x)
                    out.append(x)
            return out

        fs = _dedup(fs)
        ts = _dedup(ts)
        locs = _dedup(locs)

        floc_labels = []
        for f in fs:
            for loc in locs or ["unknown"]:
                floc_labels.append(f"{f}({loc})")

        return pd.Series({
            "treatment_marker_fluor_codes": "|".join(fs) if fs else np.nan,
            "treatment_marker_tag_codes": "|".join(ts) if ts else np.nan,
            "treatment_marker_localizations": "|".join(locs) if locs else np.nan,
            "treatment_marker_fluor_loc_labels": "|".join(floc_labels) if floc_labels else np.nan,
        })

    tx_agg = tx_markers.groupby("roi_dir", as_index=False).apply(_agg_tx)
    df_enrich = df_enrich.merge(tx_agg, how="left", on="roi_dir")

print("\nV6-08 — treatment marker rollup sample:")
print(
    df_enrich[
        [
            "roi_dir",
            "treatment_rna_rna_base_code",
            "treatment_marker_fluor_codes",
            "treatment_marker_localizations",
            "treatment_marker_fluor_loc_labels",
        ]
    ].head(20)
)

# 3) recompute all_unique_organelles / all_fluor_organelles

def _split_pipe(val):
    if pd.isna(val):
        return []
    s = str(val).strip()
    if not s:
        return []
    return [p.strip() for p in s.split("|") if p.strip()]

all_unique = []
all_fluor = []

for idx, row in df_enrich.iterrows():
    geno_locs = _split_pipe(row.get("genotype_marker_localizations", np.nan))
    tx_locs   = _split_pipe(row.get("treatment_marker_localizations", np.nan))

    locs = []
    seen_l = set()
    for x in geno_locs + tx_locs:
        if x and x not in seen_l:
            seen_l.add(x)
            locs.append(x)

    geno_floc = _split_pipe(row.get("genotype_marker_fusion_labels", np.nan))
    tx_floc   = _split_pipe(row.get("treatment_marker_fluor_loc_labels", np.nan))

    floc = []
    seen_f = set()
    for x in geno_floc + tx_floc:
        if x and x not in seen_f:
            seen_f.add(x)
            floc.append(x)

    all_unique.append("|".join(locs) if locs else np.nan)
    all_fluor.append("|".join(floc) if floc else np.nan)

df_enrich["all_unique_organelles"] = all_unique
df_enrich["all_fluor_organelles"]  = all_fluor

print("\nV6-08 — organelle recompute sample:")
print(
    df_enrich[
        [
            "roi_dir",
            "genotype_marker_localizations",
            "treatment_marker_localizations",
            "all_unique_organelles",
            "all_fluor_organelles",
        ]
    ].head(20)
)

V6-08 — RNA basecode mapping sample:
   treatment_rna_names_sheet treatment_rna_rna_base_code
0                        NaN                         NaN
1                        NaN                         NaN
2                        NaN                         NaN
3                        NaN                         NaN
4                        NaN                         NaN
5                        NaN                         NaN
6                        NaN                         NaN
7                        NaN                         NaN
8                        NaN                         NaN
9                        NaN                         NaN
10                       NaN                         NaN
11                       NaN                         NaN
12                       NaN                         NaN
13                       NaN                         NaN
14                       NaN                         NaN
15                       NaN                       

In [57]:
# V6-09 — build DB subset df_for_db

df_enrich = df_enrich.copy()

required_cols = [
    "roi_dir",
    "bruker_roi_id",
    "plate_date",
    "plate_id_filled",
    "slot_id_filled",
    "roi_index_within_slot",
    "dataset_slug",
    "experiment_folder",
    "fish_id",
    "fish_number",
    "fish_age_hpf",
    "roi_anatomy",
    "roi_tiffs",
    "date_experiment",
    "Date imaged",
    "date_mount",
    "genotype_base_codes",
    "genotype_allele_codes",
    "treatment_plasmid_plasmid_base_code",
    "treatment_rna_rna_base_code",
    "genotype_marker_fluor_codes",
    "genotype_marker_tag_codes",
    "genotype_marker_localizations",
    "genotype_marker_fusion_labels",
    "treatment_marker_fluor_codes",
    "treatment_marker_tag_codes",
    "treatment_marker_localizations",
    "treatment_marker_fluor_loc_labels",
    "all_unique_organelles",
    "all_fluor_organelles",
    "link_source",
]

# create missing columns as NaN
for c in required_cols:
    if c not in df_enrich.columns:
        df_enrich[c] = np.nan

df_for_db = df_enrich[required_cols].copy()

print("V6-09 — df_for_db shape:", df_for_db.shape)
print("V6-09 — df_for_db head:")
print(df_for_db.head(10))

# quick duplicate check
dups = df_for_db[df_for_db.duplicated("roi_dir", keep=False)]
print("\nV6-09 — duplicate roi_dir rows in df_for_db:", len(dups))
if len(dups):
    print(dups[["roi_dir", "bruker_roi_id"]].head(20))

V6-09 — df_for_db shape: (976, 31)
V6-09 — df_for_db head:
                                             roi_dir                bruker_roi_id  plate_date  plate_id_filled  slot_id_filled  roi_index_within_slot                               dataset_slug                          experiment_folder  \
0  /clusterfs/vast/abcabc/Aang_Foundation/2025072...  20250721-plate65-slot1-roi1  20250721.0             65.0             1.0                    1.0  20250721_72hpf_mrna_mSG_organelle_LLS-SIM  20250721_72hpf_mrna_mSG_organelle_LLS-SIM   
1  /clusterfs/vast/abcabc/Aang_Foundation/2025072...  20250721-plate65-slot2-roi1  20250721.0             65.0             2.0                    1.0  20250721_72hpf_mrna_mSG_organelle_LLS-SIM  20250721_72hpf_mrna_mSG_organelle_LLS-SIM   
2  /clusterfs/vast/abcabc/Aang_Foundation/2025072...  20250721-plate65-slot3-roi1  20250721.0             65.0             3.0                    1.0  20250721_72hpf_mrna_mSG_organelle_LLS-SIM  20250721_72hpf_mrna_mSG_organe

In [58]:
# V6-9b — apply organelle overrides from organelles_manual_mapping_v6.csv

from pathlib import Path
import pandas as pd

if "df_enrich" not in globals():
    raise NameError("V6-11: df_enrich not found; run earlier V6 cells first.")

ROOT = Path("/Users/davekokel/Projects/carp_v2")
BASE = ROOT / "seed_kits" / "legacy_wrangling_v2"
RAW  = BASE / "raw"

ORG_MAP_PATH = RAW / "organelles_manual_mapping_v6.csv"
print("V6-11 — organelle mapping path:", ORG_MAP_PATH)

if not ORG_MAP_PATH.exists():
    raise FileNotFoundError(f"V6-11: mapping CSV not found at {ORG_MAP_PATH}")

org_map = pd.read_csv(ORG_MAP_PATH)

# normalize slug key
def _norm_dataset_slug(s):
    if pd.isna(s):
        return None
    return str(s).strip()

org_map["dataset_slug_norm"] = org_map["dataset_slug"].apply(_norm_dataset_slug)
org_map = org_map.dropna(subset=["dataset_slug_norm"])

print("V6-11 — organelle mapping rows:", len(org_map))
print("V6-11 — organelle mapping unique dataset_slug_norm:", org_map["dataset_slug_norm"].nunique())

# ensure df_enrich has dataset_slug + organelle cols as string
df_enrich = df_enrich.copy()

for col in ["dataset_slug", "all_unique_organelles", "all_fluor_organelles"]:
    if col in df_enrich.columns:
        df_enrich[col] = df_enrich[col].astype("string")

df_enrich["dataset_slug_norm"] = df_enrich["dataset_slug"].apply(_norm_dataset_slug)

# merge overrides
df_enrich = df_enrich.merge(
    org_map[
        [
            "dataset_slug_norm",
            "all_unique_organelles_override",
            "all_fluor_organelles_override",
        ]
    ],
    how="left",
    on="dataset_slug_norm",
)

# identify rows currently missing organelles
def _is_empty_col(s: pd.Series) -> pd.Series:
    return s.isna() | (s.astype(str).str.strip().isin(["", "None", "nan", "<NA>"]))

mask_missing_org = _is_empty_col(df_enrich["all_unique_organelles"])

mask_has_override_unique = df_enrich["all_unique_organelles_override"].notna() & (
    df_enrich["all_unique_organelles_override"].astype(str).str.strip() != ""
)
mask_has_override_fluor = df_enrich["all_fluor_organelles_override"].notna() & (
    df_enrich["all_fluor_organelles_override"].astype(str).str.strip() != ""
)

to_patch_unique = mask_missing_org & mask_has_override_unique
to_patch_fluor  = _is_empty_col(df_enrich["all_fluor_organelles"]) & mask_has_override_fluor

print("\nV6-11 — organelle override patch counts:")
print("  rows currently missing all_unique_organelles:", int(mask_missing_org.sum()))
print("  rows with unique-org override available:     ", int(mask_has_override_unique.sum()))
print("  rows patched for all_unique_organelles:      ", int(to_patch_unique.sum()))
print("  rows patched for all_fluor_organelles:       ", int(to_patch_fluor.sum()))

df_enrich.loc[to_patch_unique, "all_unique_organelles"] = df_enrich.loc[
    to_patch_unique, "all_unique_organelles_override"
].astype("string")

df_enrich.loc[to_patch_fluor, "all_fluor_organelles"] = df_enrich.loc[
    to_patch_fluor, "all_fluor_organelles_override"
].astype("string")

# small sanity sample for a few key datasets
for slug in [
    "20250513_skittles",
    "20250522_skittlez",
    "20250808_mem_organelle",
    "20250828_mem_actin",
    "20250722_mrna_mito-mSG_mem-mChilada",
]:
    sub = df_enrich[df_enrich["dataset_slug"] == slug]
    if sub.empty:
        continue
    print(f"\nV6-11 — sample rows after override for dataset_slug='{slug}':")
    print(
        sub[
            [
                "roi_dir",
                "dataset_slug",
                "all_unique_organelles",
                "all_fluor_organelles",
            ]
        ].head(5)
    )

print("\nV6-11 — done. Now re-run V6-10 to recompute QC + write V6 CSVs/DB CSV.")

V6-11 — organelle mapping path: /Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling_v2/raw/organelles_manual_mapping_v6.csv
V6-11 — organelle mapping rows: 21
V6-11 — organelle mapping unique dataset_slug_norm: 21

V6-11 — organelle override patch counts:
  rows currently missing all_unique_organelles: 434
  rows with unique-org override available:      336
  rows patched for all_unique_organelles:       336
  rows patched for all_fluor_organelles:        112

V6-11 — sample rows after override for dataset_slug='20250513_skittles':
                                               roi_dir       dataset_slug all_unique_organelles                               all_fluor_organelles
566  /clusterfs/vast/abcabc/Korra_Foundation/202505...  20250513_skittles               cytosol  mKate2(unknown)|mCitrine(unknown)|Electra2(unk...
567  /clusterfs/vast/abcabc/Korra_Foundation/202505...  20250513_skittles               cytosol  mKate2(unknown)|mCitrine(unknown)|Electra2(unk...
568  /cluste

In [59]:
# V6-10 — rebuild DB subset + QC from CURRENT df_enrich (after all patches)

import pandas as pd
from pathlib import Path

if "df_enrich" not in globals():
    raise NameError("V6-10: df_enrich not found; run V6-01..V6-09/V6-11 first.")

ROOT = Path("/Users/davekokel/Projects/carp_v2")
BASE = ROOT / "seed_kits" / "legacy_wrangling_v2"
WORKING = BASE / "working"

FULL_OUT_V6 = WORKING / "legacy_imaging_annotations_v6.csv"
DB_OUT_V6   = WORKING / "legacy_imaging_annotations_for_db_v6.csv"

print("V6-10 — rebuilding DB subset from current df_enrich")
print("  df_enrich shape:", df_enrich.shape)

# ─────────────────────────────────────────────
# 1) Build df_for_db from df_enrich
# ─────────────────────────────────────────────

cols_db = [
    "roi_dir",
    "bruker_roi_id",
    "plate_date",
    "plate_id_filled",
    "slot_id_filled",
    "roi_index_within_slot",
    "dataset_slug",
    "experiment_folder",
    "fish_id",
    "fish_number",
    "fish_age_hpf",
    "roi_anatomy",
    "roi_tiffs",
    "date_experiment",
    "Date imaged",
    "date_mount",
    "genotype_base_codes",
    "genotype_allele_codes",
    "treatment_plasmid_plasmid_base_code",
    "treatment_rna_rna_base_code",
    "genotype_marker_fluor_codes",
    "genotype_marker_tag_codes",
    "genotype_marker_localizations",
    "genotype_marker_fusion_labels",
    "all_unique_organelles",
    "all_fluor_organelles",
    "link_source",
]

missing_cols = [c for c in cols_db if c not in df_enrich.columns]
if missing_cols:
    raise KeyError(f"V6-10: df_enrich is missing expected DB columns: {missing_cols}")

df_for_db = df_enrich[cols_db].copy()

# make sure organelle columns are string-ish
for col in ["all_unique_organelles", "all_fluor_organelles"]:
    df_for_db[col] = df_for_db[col].astype("string")

print("\nV6-10 — df_for_db shape:", df_for_db.shape)
print("V6-10 — unique roi_dir in df_for_db:", df_for_db["roi_dir"].nunique())

# ─────────────────────────────────────────────
# 2) Organelles QC on df_for_db
# ─────────────────────────────────────────────

def _is_empty_org(s: pd.Series) -> pd.Series:
    return s.isna() | s.astype(str).str.strip().isin(["", "None", "nan", "<NA>"])

missing_mask = _is_empty_org(df_for_db["all_unique_organelles"])
n_total   = len(df_for_db)
n_missing = int(missing_mask.sum())
n_present = n_total - n_missing

print("\nV6-10 — organelle coverage (df_for_db):")
print("  total ROIs:                   ", n_total)
print("  ROIs with organelles present: ", n_present)
print("  ROIs missing organelles:      ", n_missing)

print("\nV6-10 — missing organelles by link_source:")
print(df_for_db.loc[missing_mask, "link_source"].value_counts(dropna=False))

print("\nV6-10 — top datasets among no-org rows:")
print(
    df_for_db.loc[missing_mask, "dataset_slug"]
    .value_counts()
    .head(20)
)

# ─────────────────────────────────────────────
# 3) Write outputs
# ─────────────────────────────────────────────

df_enrich.to_csv(FULL_OUT_V6, index=False)
df_for_db.to_csv(DB_OUT_V6, index=False)

print("\nV6-10 — wrote:")
print("  FULL_OUT_V6:", FULL_OUT_V6)
print("  DB_OUT_V6:  ", DB_OUT_V6)

V6-10 — rebuilding DB subset from current df_enrich
  df_enrich shape: (976, 60)

V6-10 — df_for_db shape: (976, 27)
V6-10 — unique roi_dir in df_for_db: 976

V6-10 — organelle coverage (df_for_db):
  total ROIs:                    976
  ROIs with organelles present:  878
  ROIs missing organelles:       98

V6-10 — missing organelles by link_source:
link_source
sheet         85
unmatched     12
date_match     1
Name: count, dtype: int64

V6-10 — top datasets among no-org rows:
dataset_slug
20250819_mem_ER                              8
20250429_mem_cytosol                         8
20250805_mito_mem-halo                       7
20251021_mem-mito_v2                         7
20250723_mem-halo_ER-mSG                     6
20250903_lifeact-mSG                         6
20250721_72hpf_mrna_mSG_organelle_LLS-SIM    5
20250805_lifeact_mem-halo                    5
20250624_skittlez_Issac_x_Ken                5
20250521_skittles_no-membrane                5
20251028_mem-peroxi2              